The code below is to get the data and turn it into csv files

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf

pd.set_option('display.max_columns', None)

start = "2018-01-01"
end = "2024-01-01"
tickers = ["SPY","QQQ","TLT"]

for ticker in tickers:
    df = yf.download(ticker,start,end)

    #isNull() makes the df into 1 and 0 where 1 is null. The first sum gets the number of 1 in each column.The second sum sums the number in each column
    num_missing = df.isnull().sum().sum()
    print(f"Total Missing: {num_missing}")

    if num_missing>0:
        #This fills the missing values. ffill() fills it with prev value. bfill() fills with value ahead if there is nothing prev.
        df = df.ffill().bfill()
    
    #Gets rid of the multiIndex which can be annoying. We get rid 2nd layer of the columns
    if isinstance(df.columns,pd.MultiIndex):
        df.columns = df.columns.droplevel(1)

    
    filename = f"{ticker}_raw.csv"
    df.to_csv(filename)






[*********************100%***********************]  1 of 1 completed


Total Missing: 0


[*********************100%***********************]  1 of 1 completed


Total Missing: 0


[*********************100%***********************]  1 of 1 completed

Total Missing: 0


In [2]:
def get_indicators(df):
    """
    Takes a raw OHLCV dataframe, shifts it to avoid lookahead bias, 
    and adds RSI, MACD, and Bollinger Bands.
    """
    df = df.copy() 
    #shifts the day one down. So for example tuesday only has the clost information available on monday. No look ahead bias
    df['Past_close'] = df['Close'].shift(1)

    #Relative Strength Index(14 day rolling window)
    #Delta is a df that is the current day subtracted from previous day
    delta = df['Past_close'].diff()
    gain = (delta.where(delta>0,0)).rolling(window=14).mean()
    loss = (-delta.where(delta<0,0)).rolling(window=14).mean()
    rs = gain/loss
    df['RSI'] = 100 - (100 / (1 + rs))


    #Bollinger Bands(Usually a 20-day rolling mean, plus/minus 2 standard deviations)
    df['BB_Mid'] = df['Past_close'].rolling(window=20).mean()
    bb_std = df['Past_close'].rolling(window=20).std()
    df['BB_Upper'] = df['BB_Mid'] + (2 * bb_std)
    df['BB_Lower'] = df['BB_Mid'] - (2 * bb_std)


    #Moving Average Convergence Divergence(Difference between 12-day EMA and 26-day EMA)
    ema_12 = df['Past_close'].ewm(span=12, adjust=False).mean()
    ema_26 = df['Past_close'].ewm(span=26,adjust=False).mean()
    df['MACD'] = ema_12 - ema_26
    #When MACD is above the signal line, it means the trend is going upward. When MACD is below the signal line it means trend is going down
    df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    
    #Gets rid of NaN
    df.dropna(inplace=True) 

    return df




The code below is to get the data seperated for training

In [3]:
import pickle

#To hold all of our dataframes
all_dfs = []

tickers = ["SPY","QQQ","TLT"]

for tick in tickers:
    df_raw = pd.read_csv(f"{tick}_raw.csv", index_col=0, parse_dates=True)
    df_engineered = get_indicators(df_raw)

    df_engineered['Ticker'] = tick

    #These are for when training the model
    df_engineered['weight'] = 0.0
    df_engineered['regime_prob'] = 0.0

    all_dfs.append(df_engineered)

all_dfs = pd.concat(all_dfs)

print(all_dfs[all_dfs.index == all_dfs.index[0]][["Ticker","Close"]])

#This below will be the training, validating, and testing sets below
#Train is 2018-2021, Validate will be 2022-2023, test will be 2023-2024
train_data = all_dfs.loc[(all_dfs.index<'2022-01-01')]
val_data = all_dfs.loc[(all_dfs.index<'2023-01-01') & (all_dfs.index>='2022-01-01')]
test_data =  all_dfs.loc[(all_dfs.index>='2023-01-01')]

print(val_data[val_data.index == val_data.index[0]][["Ticker", "Close"]])

display(val_data)
print(f"Train Shape: {train_data.shape}")
#There are 753 rows because three tickers 251*3 = 753
print(f"Val Shape: {val_data.shape}")
print(f"Test Shape: {test_data.shape}")

#Turn to dict so it is easier for RL gym env
output = {
    'train': train_data,
    'val': val_data,
    'test':test_data
}

#Turn to pickle because it is dict
with open('engineered.pkl', 'wb') as f:
    pickle.dump(output, f)







    





           Ticker       Close
Date                         
2018-01-31    SPY  248.118744
2018-01-31    QQQ  160.563019
2018-01-31    TLT   97.272308
           Ticker       Close
Date                         
2022-01-03    SPY  450.644440
2022-01-03    QQQ  391.186218
2022-01-03    TLT  123.909157


,Close,High,Low,Open,Volume,Past_close,RSI,BB_Mid,BB_Upper,BB_Lower,MACD,MACD_signal,Ticker,weight,regime_prob
Date,,,,,,,,,,,,,,,
2022-01-03,450.644440,450.776522,447.003149,449.314322,72668200,448.050262,55.399231,440.386363,454.484680,426.288047,3.877230,2.789352,SPY,0.0,0.0
2022-01-04,450.493469,452.785815,448.635082,452.068865,71178700,450.644440,62.065008,441.606923,454.739799,428.474047,4.088957,3.049273,SPY,0.0,0.0
2022-01-05,441.843048,450.899148,441.748708,450.125599,104538900,450.493469,65.844783,442.567538,455.325510,429.809567,4.196199,3.278658,SPY,0.0,0.0
2022-01-06,441.427979,444.144811,439.060178,441.380823,86858900,441.843048,49.448009,442.649582,455.364539,429.934625,3.542338,3.331394,SPY,0.0,0.0
2022-01-07,439.682709,442.616520,438.324293,441.437342,85111600,441.427979,53.242505,442.652589,455.366298,429.938881,2.956575,3.256430,SPY,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-12-23,89.788857,90.360139,89.665804,90.157995,15408900,91.124802,40.093126,92.754993,96.881570,88.628416,0.994749,1.576330,TLT,0.0,0.0
2022-12-27,88.013474,88.830855,87.899218,88.321089,26475700,89.788857,40.033590,92.744100,96.902252,88.585948,0.705512,1.402166,TLT,0.0,0.0
2022-12-28,87.494896,88.575944,87.319111,88.461689,17302900,88.013474,30.256852,92.631754,97.175950,88.087559,0.329235,1.187580,TLT,0.0,0.0


Train Shape: (2964, 15)
Val Shape: (753, 15)
Test Shape: (750, 15)


In [4]:
import pandas as pd

spy = pd.read_csv("SPY_raw.csv")
qqq = pd.read_csv("QQQ_raw.csv")
tlt = pd.read_csv("TLT_raw.csv")

print("SPY:", spy["Close"].head())
print("QQQ:", qqq["Close"].head())
print("TLT:", tlt["Close"].head())

SPY: 0    236.562210
1    238.058426
2    239.061844
3    240.654968
4    241.095032
Name: Close, dtype: float64
QQQ: 0    150.222107
1    151.681808
2    151.947189
3    153.473236
4    154.070328
Name: Close, dtype: float64
TLT: 0    99.459824
1    99.935387
2    99.919518
3    99.634209
4    99.570793
Name: Close, dtype: float64
